# Trader research exploration

This notebook walks through the five frames written by `trader dump-frames`.

**Prerequisites:**

1. `poetry install --with dev,alpaca,docs` (the `docs` group brings matplotlib).
2. Alpaca credentials in `.env` or exported.
3. Run a dump first, e.g.
   ```
   trader dump-frames --symbols AAPL SPY BTC/USD --timeframe 1d --bars 500 --out-dir data/research
   ```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from trader.research import load_frames, plot_equity_curve, summarize_trades

frames = load_frames(Path('../data/research'))
print({name: getattr(frames, name).shape for name in ('ohlcv', 'indicators', 'signals', 'trades', 'equity')})

## 1. Raw OHLCV

In [ ]:
frames.ohlcv.groupby('symbol').tail(3)

## 2. Indicator frame

Show a slice of the trend / volatility / momentum columns for the most recent rows.

In [ ]:
cols = ['symbol', 'time', 'close', 'ema_20', 'ema_50', 'trend_adx', 'momentum_rsi', 'atr_pct', 'trend_regime']
frames.indicators.groupby('symbol').tail(3)[cols]

## 3. Signals

Distribution of signal values per symbol.

In [ ]:
frames.signals.groupby('symbol')['signal'].value_counts().unstack(fill_value=0)

## 4. Trade summary

In [ ]:
summarize_trades(frames.trades)

## 5. Equity curve

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
plot_equity_curve(frames.equity, ax=ax)
ax.set_title('Portfolio equity curve')
plt.show()

In [ ]:
# Per-symbol overlay
fig, ax = plt.subplots(figsize=(11, 4))
symbols = [s for s in frames.equity['symbol'].unique() if s != 'PORTFOLIO']
plot_equity_curve(frames.equity, symbols=symbols, ax=ax)
ax.set_title('Per-symbol equity curves')
plt.show()